# 第10回：Metropolis法で二つの谷を調べる
「ランタイム → すべてのセルを実行」で計算します。GPUは不要です。演習20分、採点・提出なし。

今回の模型は無次元のエネルギー $U(x)=(x^2-1)^2+0.25x$ です。`temperature`は熱エネルギー $k_BT$ を表します。タンパク質の定量モデルではありません。

**見る順序**：提案・受理のコード → 軌跡 → 理論密度との比較 → 谷の確率と相関。

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import json
from pathlib import Path

def energy(x):
    return (x*x-1)**2 + 0.25*x

def metropolis(n=60000, temperature=0.6, sigma=0.7, x0=-1.0, seed=42):
    assert n > 100 and temperature > 0 and sigma > 0
    rng = np.random.default_rng(seed)
    x = float(x0)
    chain = np.empty(n)
    accepted = 0
    for t in range(n):
        y = x + sigma*rng.normal()  # 対称な提案
        log_alpha = min(0.0, -(energy(y)-energy(x))/temperature)
        if np.log(rng.random()) < log_alpha:
            x = y
            accepted += 1
        chain[t] = x  # 棄却したときも現在値を記録する
    return chain, accepted/n


In [ ]:
# まずはこの設定のまま実行。その後、一つだけ変えます。
temperature = 0.6
sigma = 0.7
n_steps = 60000
burn_in = 5000
seed = 42
x0 = -1.0
chain, acceptance = metropolis(n_steps, temperature, sigma, x0, seed)
samples = chain[burn_in:]
grid = np.linspace(-3, 3, 12001)
weights = np.exp(-(energy(grid)-energy(grid).min())/temperature)
integrate = np.trapezoid if hasattr(np, 'trapezoid') else np.trapz
density = weights/integrate(weights, grid)
# 右側の確率は x>0 で定義。離散化した台形積分と標本の両方で同じ境界を使う。
p_right_theory = integrate(density[grid>=0], grid[grid>=0])
mean_theory = integrate(grid*density, grid)
print(f'受理率={acceptance:.3f}, 平均={samples.mean():.3f} / 理論={mean_theory:.3f}')
print(f'P(x>0)={np.mean(samples>0):.3f} / 理論={p_right_theory:.3f}')
fig, ax = plt.subplots(1,3,figsize=(16,4))
ax[0].plot(grid,energy(grid)); ax[0].set(xlim=(-2,2),ylim=(-.4,4),title='Energy',xlabel='x')
ax[1].plot(chain[:5000],lw=.6);ax[1].set(title='Trace (MC steps, not physical time)',xlabel='step',ylabel='x')
ax[2].hist(samples,bins=70,density=True,alpha=.5,label='samples')
ax[2].plot(grid,density,label='Boltzmann');ax[2].set(xlim=(-2,2),title='Probability density',xlabel='x');ax[2].legend()
plt.tight_layout();plt.show()


## 相関と再現性
隣り合う標本は独立ではありません。谷の所属を表す $I(x>0)$ の相関も調べます。
有効標本数は目安で、未訪問の谷がないことを保証しません。初期値を+1へ変えた実行とも比べます。

In [ ]:
def autocorrelation(x, max_lag=1000):
    z = np.asarray(x,dtype=float)-np.mean(x)
    f = np.fft.rfft(z,n=2*len(z))
    c = np.fft.irfft(f*np.conj(f))[:max_lag+1]
    c /= np.arange(len(z),len(z)-len(c),-1)
    return c/c[0] if c[0]>0 else np.full(len(c),np.nan)
for name, values in [('x',samples),('I(x>0)',samples>0)]:
    rho = autocorrelation(values)
    # 最初の非正の値で切る簡易推定。信頼区間の厳密な推定ではない。
    stop = np.flatnonzero(rho[1:]<=0)
    cut = int(stop[0]+1) if len(stop) else len(rho)
    tau_factor = 1+2*np.sum(rho[1:cut])
    print(name, 'N_eff (approx)=', round(len(values)/tau_factor) if np.isfinite(tau_factor) else '未推定')
    plt.plot(rho[:300],label=name)
plt.xlabel('lag');plt.ylabel('autocorrelation');plt.legend();plt.show()
# 独立なseed、別の初期値で比較。これは収束の証明ではなく診断。
for xstart, s in [(-1,7),(1,8)]:
    c,a = metropolis(n_steps,temperature,sigma,xstart,s)
    print('x0=',xstart,'seed=',s,'P_right=',float(np.mean(c[burn_in:]>0)),'accept=',round(a,3))


## 条件を変える
- `sigma=0.1`：受理率が高くても谷を往復できるか。
- `sigma=2.0`：棄却が増えたときの軌跡を観察。
- `temperature=0.25`：障壁を越える頻度と、平衡での谷の割合を区別。

条件を変えたら設定セル以降を再実行します。棄却を除く変更は行わないでください。


In [ ]:
config = dict(temperature=temperature,sigma=sigma,n_steps=n_steps,burn_in=burn_in,seed=seed,x0=x0)
Path('bioinfo10_conditions.json').write_text(json.dumps(config,indent=2))
np.savez_compressed('bioinfo10_samples.npz',chain=chain)
print('保存: bioinfo10_conditions.json, bioinfo10_samples.npz')


## 参考
Metropolis et al. (1953), Equation of State Calculations by Fast Computing Machines. https://doi.org/10.1063/1.1699114

Hastings (1970), Monte Carlo sampling methods using Markov chains and their applications. https://doi.org/10.1093/biomet/57.1.97